# 03 — Model Training

## Architecture
```
Linear(n_features → 64) → ReLU → Dropout(0.1)
Linear(64 → 32) → ReLU
Linear(32 → 1)
```

## Training protocol
- Loss: Huber (δ=1.0) — robust to outlier bars
- Optimizer: Adam (lr=1e-3)
- Scheduler: ReduceLROnPlateau (patience=5, factor=0.5)
- Early stopping: patience=10 on val MAE
- Temporal split: 65% train / 15% val / 20% test
- Scaler fitted on train only (no data leakage)

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from features import FEATURE_NAMES
from pipeline import build_full_dataset
from train import train
from viz import plot_training_history, FIGURES_DIR

In [ ]:
data, proxy_all, split = build_full_dataset()
print(f'Train: {len(split.X_train):,}  Val: {len(split.X_val):,}  Test: {len(split.X_test):,}')

In [ ]:
model, history = train(
    split,
    n_features=len(FEATURE_NAMES),
    epochs=100,
    batch_size=256,
    lr=1e-3,
    patience=10,
    seed=42,
    verbose=True,
)
print(f'\nBest epoch: {history["best_epoch"]}  |  Best val MAE: {history["best_val_mae"]:.4f} bps')

In [ ]:
fig = plot_training_history(history, save_as='training_history.png')
plt.show()

In [ ]:
# Save split metadata for notebook 04
import pickle
results_dir = pathlib.Path.cwd().parent / 'results'
results_dir.mkdir(exist_ok=True)
with open(results_dir / 'split_metadata.pkl', 'wb') as f:
    pickle.dump({'n_train': len(split.X_train), 'n_val': len(split.X_val), 'n_test': len(split.X_test)}, f)
print('Checkpoint and metadata saved.')